In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import os

# На всякий случай помним, что на Kaggle данные лежат в /kaggle/input/avito-demand-prediction/
# А результаты мы пишем в текущую рабочую директорию (/kaggle/working/)
PATH = "/kaggle/input/competitions/avito-demand-prediction"

print("Загрузка данных...")
train = pd.read_csv(os.path.join(PATH, 'train.csv'), parse_dates=['activation_date'])
test = pd.read_csv(os.path.join(PATH, 'test.csv'), parse_dates=['activation_date'])

print(f"Размер train: {train.shape}")
print(f"Размер test: {test.shape}")

# Посмотрим на первые строки и пропуски
print(train.info())

In [ ]:
# Объединяем для однородной обработки категорий
target = train['deal_probability']
train_len = len(train)
df = pd.concat([train.drop(['deal_probability'], axis=1), test], axis=0).reset_index(drop=True)

# 1. Работа с датой
df['weekday'] = df['activation_date'].dt.weekday
df['day'] = df['activation_date'].dt.day

# 2. Простейшее заполнение пропусков в цене
df['price'] = df['price'].fillna(df['price'].median())

# 3. Кодируем категориальные признаки
cat_cols = ['user_type', 'parent_category_name', 'category_name', 'region', 'city']
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Выделяем числовые и закодированные признаки для бейзлайна
features = ['price', 'item_seq_number', 'user_type', 'parent_category_name', 
            'category_name', 'region', 'city', 'weekday', 'day']

X_train = df.iloc[:train_len][features]
X_test = df.iloc[train_len:][features]
y_train = target

# Разделяем на train/val для валидации
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [ ]:
# Создаем датасеты для LightGBM
lgb_train = lgb.Dataset(X_tr, y_tr)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.1,
    'num_leaves': 31,
    'seed': 42,
    'verbose': -1
}

print("Обучение LightGBM...")
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)

# Проверяем локальный скор
val_preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, val_preds))
print(f"Локальный RMSE на валидации: {rmse:.5f}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

print("Обработка текстовых признаков...")

# Заполняем пропуски пустыми строками
df['title'] = df['title'].fillna('').astype(str)
df['description'] = df['description'].fillna('').astype(str)

# 1. Считаем простые длины текстов (в символах и словах)
df['title_len'] = df['title'].apply(len)
df['description_len'] = df['description'].apply(len)
df['title_word_count'] = df['title'].apply(lambda x: len(x.split()))
df['description_word_count'] = df['description'].apply(lambda x: len(x.split()))

# 2. Настраиваем TF-IDF + SVD для Title
print("Выделяем фичи из title...")
tfidf_title = TfidfVectorizer(max_features=5000, stop_words=None) # Для русского языка можно без стоп-слов на базовом этапе
svd_title = TruncatedSVD(n_components=5, random_state=42)

title_tfidf = tfidf_title.fit_transform(df['title'])
title_svd = svd_title.fit_transform(title_tfidf)

# 3. Настраиваем TF-IDF + SVD для Description
print("Выделяем фичи из description...")
tfidf_desc = TfidfVectorizer(max_features=10000, stop_words=None)
svd_desc = TruncatedSVD(n_components=5, random_state=42)

desc_tfidf = tfidf_desc.fit_transform(df['description'])
desc_svd = svd_desc.fit_transform(desc_tfidf)

# Переносим полученные SVD-компоненты в наш основной датафрейм
for i in range(5):
    df[f'title_svd_{i}'] = title_svd[:, i]
    df[f'desc_svd_{i}'] = desc_svd[:, i]

print("Текстовые фичи успешно добавлены!")

In [ ]:
# Расширяем список фич
text_features = ['title_len', 'description_len', 'title_word_count', 'description_word_count'] + \
                [f'title_svd_{i}' for i in range(5)] + \
                [f'desc_svd_{i}' for i in range(5)]

all_features = features + text_features

# Пересобираем матрицы
X_train_new = df.iloc[:train_len][all_features]
y_train_new = target

X_tr_n, X_val_n, y_tr_n, y_val_n = train_test_split(X_train_new, y_train_new, test_size=0.2, random_state=42)

lgb_train_n = lgb.Dataset(X_tr_n, y_tr_n)
lgb_val_n = lgb.Dataset(X_val_n, y_val_n, reference=lgb_train_n)

print("Обучение новой модели с текстовыми фичами...")
model_n = lgb.train(
    params,
    lgb_train_n,
    num_boost_round=1000,
    valid_sets=[lgb_train_n, lgb_val_n],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)

val_preds_n = model_n.predict(X_val_n)
rmse_n = np.sqrt(mean_squared_error(y_val_n, val_preds_n))
print(f"Новый локальный RMSE на валидации: {rmse_n:.5f}")

In [ ]:
print("Генерация агрегированных признаков...")

# 1. Группировка по категориям: считаем среднюю цену и стандартное отклонение
cat_price_stats = df.groupby('category_name')['price'].agg(['mean', 'std']).reset_index()
cat_price_stats.columns = ['category_name', 'cat_price_mean', 'cat_price_std']

# 2. Группировка по регионам и категориям вместе
reg_cat_price_stats = df.groupby(['region', 'category_name'])['price'].agg(['mean']).reset_index()
reg_cat_price_stats.columns = ['region', 'category_name', 'reg_cat_price_mean']

# Объединяем эти данные с основным датафреймом
df = pd.merge(df, cat_price_stats, on='category_name', how='left')
df = pd.merge(df, reg_cat_price_stats, on=['region', 'category_name'], how='left')

# 3. Считаем относительные фичи (насколько цена отличается от средней)
df['price_to_mean_cat'] = df['price'] / (df['cat_price_mean'] + 1)
df['price_to_mean_reg_cat'] = df['price'] / (df['reg_cat_price_mean'] + 1)

# Заполняем возможные пустоты, которые могли появиться при делении или расчете std
df['cat_price_std'] = df['cat_price_std'].fillna(0)

print("Агрегации успешно добавлены!")

In [ ]:
# Добавляем новые фичи к общему списку
agg_features = ['cat_price_mean', 'cat_price_std', 'reg_cat_price_mean', 'price_to_mean_cat', 'price_to_mean_reg_cat']
final_features = all_features + agg_features

# Пересобираем матрицы
X_train_final = df.iloc[:train_len][final_features]
y_train_final = target

X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(X_train_final, y_train_final, test_size=0.2, random_state=42)

lgb_train_f = lgb.Dataset(X_tr_f, y_tr_f)
lgb_val_f = lgb.Dataset(X_val_f, y_val_f, reference=lgb_train_f)

print("Обучение модели с агрегациями...")
model_f = lgb.train(
    params,
    lgb_train_f,
    num_boost_round=1500, # Увеличили количество раундов
    valid_sets=[lgb_train_f, lgb_val_f],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)

val_preds_f = model_f.predict(X_val_f)
rmse_f = np.sqrt(mean_squared_error(y_val_f, val_preds_f))
print(f"Итоговый локальный RMSE после агрегаций: {rmse_f:.5f}")

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import zipfile
import io
from tqdm import tqdm
from sklearn.decomposition import IncrementalPCA

print("Проверяем доступность GPU...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")

# 1. Загружаем предобученную модель (ResNet18 — она быстрая и легкая)
# Отсекаем последний классификационный слой, чтобы получить чистые фичи (512 чисел)
resnet = models.resnet18(pretrained=True)
model_features = torch.nn.Sequential(*(list(resnet.children())[:-1]))
model_features = model_features.to(device)
model_features.eval()

# 2. Настраиваем трансформацию картинок под стандарт ResNet (размер 224x224 + нормализация)
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Путь к архиву с картинками (скорректируй, если у тебя другой путь в панели справа)
ZIP_PATH = "/kaggle/input/competitions/avito-demand-prediction/train_jpg.zip"

print("Начинаем тестовое извлечение эмбеддингов...")

In [ ]:
import numpy as np
from sklearn.decomposition import IncrementalPCA

# 1. Создаем базовый признак: есть ли вообще картинка
df['has_image'] = df['image'].notnull().astype(int)

# 2. Отбираем объявления, у которых картинки есть, чтобы обработать их батчами
# Для прототипа ограничимся первыми 30 000 картинок, чтобы код отработал за пару минут
sub_df = df[df['has_image'] == 1].head(30000).copy()
image_ids = sub_df['image'].values

# Настраиваем IPCA для сжатия 512 признаков от ResNet до 3 главных компонент
ipca = IncrementalPCA(n_components=3)
features_list = []
valid_indices = []

print("Открываем архив и извлекаем эмбеддинги...")
batch_size = 64
current_batch = []

with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    # Названия файлов в архиве обычно выглядят как 'data/competition_files/train_jpg/{image_id}.jpg'
    # Давай сформируем список путей. Убедись, что маска пути совпадает с архивом (можно проверить через archive.namelist()[:5])
    
    for idx, img_id in enumerate(tqdm(image_ids)):
        img_name = f"data/competition_files/train_jpg/{img_id}.jpg"
        
        try:
            # Читаем файл из zip без распаковки на диск
            img_data = archive.read(img_name)
            img = Image.open(io.BytesIO(img_data)).convert('RGB')
            tensor = preprocess(img)
            current_batch.append(tensor)
            valid_indices.append(sub_df.index[idx]) # запоминаем глобальный индекс строки в df
        except KeyError:
            # Если конкретной картинки почему-то нет в архиве, пропускаем
            continue
            
        if len(current_batch) == batch_size or (idx == len(image_ids) - 1 and len(current_batch) > 0):
            # Собираем батч в один тензор и отправляем на GPU
            batch_tensor = torch.stack(current_batch).to(device)
            
            with torch.no_grad():
                # Прогоняем через ResNet и убираем лишние размерности (squeeze)
                batch_features = model_features(batch_tensor).squeeze().cpu().numpy()
                
            if len(current_batch) == 1:
                batch_features = batch_features.reshape(1, -1)
                
            features_list.append(batch_features)
            current_batch = []

# Объединяем все извлеченные фичи
if len(features_list) > 0:
    all_img_features = np.vstack(features_list)
    print(f"Извлечено эмбеддингов: {all_img_features.shape}")
    
    # Сжимаем 512 признаков до 3 компонент
    compressed_features = ipca.fit_transform(all_img_features)
    
    # Инициализируем новые колонки в основном датафрейме нулями
    for i in range(3):
        df[f'img_pca_{i}'] = 0.0
        
    # Записываем сжатые фичи картинок строго в те строки, где они были извлечены
    df.loc[valid_indices, ['img_pca_0', 'img_pca_1', 'img_pca_2']] = compressed_features
    print("Эмбеддинги успешно сжаты и добавлены в датафрейм!")
else:
    print("Ошибка: не удалось прочитать картинки. Проверь пути внутри архива.")

In [ ]:
# Добавляем новые признаки к финальному списку
image_features = ['has_image', 'img_pca_0', 'img_pca_1', 'img_pca_2']
final_features_with_img = final_features + image_features

# Пересобираем матрицы для обучения
X_train_img = df.iloc[:train_len][final_features_with_img]
y_train_img = target

X_tr_i, X_val_i, y_tr_i, y_val_i = train_test_split(X_train_img, y_train_img, test_size=0.2, random_state=42)

lgb_train_i = lgb.Dataset(X_tr_i, y_tr_i)
lgb_val_i = lgb.Dataset(X_val_i, y_val_i, reference=lgb_train_i)

print("Обучение модели с добавлением эмбеддингов изображений...")
model_i = lgb.train(
    params,
    lgb_train_i,
    num_boost_round=1500,
    valid_sets=[lgb_train_i, lgb_val_i],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)

val_preds_i = model_i.predict(X_val_i)
rmse_i = np.sqrt(mean_squared_error(y_val_i, val_preds_i))
print(f"RMSE после интеграции картинок: {rmse_i:.5f}")

In [ ]:
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

print("Запуск продвинутого текстового анализа...")

# 1. Вытаскиваем мета-фичи из сырого текста
print("Генерируем мета-признаки...")

# Количество знаков препинания и восклицаний
df['title_excl_count'] = df['title'].apply(lambda x: x.count('!'))
df['desc_excl_count'] = df['description'].apply(lambda x: x.count('!'))
df['desc_punct_count'] = df['description'].apply(lambda x: sum(1 for char in x if char in string.punctuation))

# Количество заглавных букв (КАПС) и цифр
df['title_upper_ratio'] = df['title'].apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1))
df['desc_digit_count'] = df['description'].apply(lambda x: sum(1 for c in x if c.isdigit()))

# 2. Настраиваем мощный TF-IDF + SVD для Title (15 компонент)
print("Выделяем n-граммы и SVD для Title...")
# sublinear_tf сглаживает аномально длинные тексты
tfidf_title_adv = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), sublinear_tf=True)
svd_title_adv = TruncatedSVD(n_components=15, random_state=42)

title_tfidf_adv = tfidf_title_adv.fit_transform(df['title'])
title_svd_adv = svd_title_adv.fit_transform(title_tfidf_adv)

# 3. Настраиваем мощный TF-IDF + SVD для Description (25 компонент)
print("Выделяем n-граммы и SVD для Description...")
tfidf_desc_adv = TfidfVectorizer(max_features=35000, ngram_range=(1, 2), sublinear_tf=True)
svd_desc_adv = TruncatedSVD(n_components=25, random_state=42)

desc_tfidf_adv = tfidf_desc_adv.fit_transform(df['description'])
desc_svd_adv = svd_desc_adv.fit_transform(desc_tfidf_adv)

# Записываем новые компоненты в DataFrame
for i in range(15):
    df[f'title_svd_adv_{i}'] = title_svd_adv[:, i]

for i in range(25):
    df[f'desc_svd_adv_{i}'] = desc_svd_adv[:, i]

print("Продвинутые текстовые фичи готовы!")

In [ ]:
# Собираем полный список новых текстовых фич
adv_text_features = [
    'title_excl_count', 'desc_excl_count', 'desc_punct_count', 
    'title_upper_ratio', 'desc_digit_count'
] + [f'title_svd_adv_{i}' for i in range(15)] + [f'desc_svd_adv_{i}' for i in range(25)]

# Наш итоговый мега-список признаков (базовые + агрегации + старый текст заменяем новым)
# Если фичи картинок (img_features) уже в df, добавь их сюда: + ['has_image', 'img_pca_0', 'img_pca_1', 'img_pca_2']
mega_features = features + agg_features + adv_text_features

# Добавляем фичи картинок, только если они созданы в df
img_cols = ['has_image', 'img_pca_0', 'img_pca_1', 'img_pca_2']
if all(col in df.columns for col in img_cols):
    mega_features += img_cols
    print("Фичи картинок успешно подключены к общему пулу!")

# Нарезаем выборки
X_train_mega = df.iloc[:train_len][mega_features]
y_train_mega = target

X_tr_m, X_val_m, y_tr_m, y_val_m = train_test_split(X_train_mega, y_train_mega, test_size=0.2, random_state=42)

lgb_train_m = lgb.Dataset(X_tr_m, y_tr_m)
lgb_val_m = lgb.Dataset(X_val_m, y_val_m, reference=lgb_train_m)

print(f"Итоговое количество признаков для модели: {len(mega_features)}")
print("Запуск финального обучения...")

model_mega = lgb.train(
    params,
    lgb_train_m,
    num_boost_round=1500,
    valid_sets=[lgb_train_m, lgb_val_m],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)

val_preds_m = model_mega.predict(X_val_m)
rmse_m = np.sqrt(mean_squared_error(y_val_m, val_preds_m))
print(f"Финальный RMSE после тюнинга текста: {rmse_m:.5f}")

In [ ]:
from sklearn.model_selection import KFold

print("Запуск продвинутого кодирования таргета (Target Encoding)...")

# Создаем колонки под новые признаки и заполняем их глобальным средним таргетом
global_mean = target.mean()
te_cols = ['category_name', 'region', 'city', 'user_type']

for col in te_cols:
    df[f'{col}_te'] = global_mean

# Выделяем отдельно трейн-часть датафрейма для KFold расчетa
train_df = df.iloc[:train_len].copy()
train_df['target'] = target.values

# Настраиваем 5-фолдовую схему для расчета средних без заглядывания в будущее
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf.split(train_df):
    X_tr_fold = train_df.iloc[train_idx]
    X_val_fold = train_df.iloc[val_idx]
    
    for col in te_cols:
        # Считаем среднее на обучающих фолдах
        mean_val = X_tr_fold.groupby(col)['target'].mean()
        # Маппим на валидационный фолд
        train_df.iloc[val_idx, train_df.columns.get_loc(f'{col}_te')] = X_val_fold[col].map(mean_val)

# Для тестовой (и валидационной общей) выборки считаем среднее по всему трейну
for col in te_cols:
    full_mean = train_df.groupby(col)['target'].mean()
    df[f'{col}_te'] = df[col].map(full_mean).fillna(global_mean)

# Возвращаем OOF-расчитанные значения обратно в общий df для трейн-части
df.iloc[:train_len, [df.columns.get_loc(f'{col}_te') for col in te_cols]] = train_df[[f'{col}_te' for col in te_cols]].values

# --- Дополнительно: Фичи истории пользователя ---
print("Генерируем признаки активности пользователей...")
# Сколько всего объявлений выставил этот пользователь за всё время (популярность юзера)
user_counts = df.groupby('user_id').size().reset_index(name='user_active_count')
df = pd.merge(df, user_counts, on='user_id', how='left')

# Отношение текущего номера объявления к общему числу объявлений юзера
df['user_item_ratio'] = df['item_seq_number'] / (df['user_active_count'] + 1)

print("Продвинутые фичи успешно сгенерированы!")

In [ ]:
# Обновляем пулл фич
te_features = [f'{col}_te' for col in te_cols] + ['user_active_count', 'user_item_ratio']
mega_features_v2 = mega_features + te_features

# Пересобираем матрицы
X_train_v2 = df.iloc[:train_len][mega_features_v2]
y_train_v2 = target

X_tr_v2, X_val_v2, y_tr_v2, y_val_v2 = train_test_split(X_train_v2, y_train_v2, test_size=0.2, random_state=42)

lgb_train_v2 = lgb.Dataset(X_tr_v2, y_tr_v2)
lgb_val_v2 = lgb.Dataset(X_val_v2, y_val_v2, reference=lgb_train_v2)

# Обновляем параметры для более глубокого обучения
params_v2 = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05, # Снизили в 2 раза
    'num_leaves': 63,      # Увеличили сложность дерева с 31 до 63
    'feature_fraction': 0.8, # Модель будет брать 80% случайных фич на дерево (защита от оверфита)
    'seed': 42,
    'verbose': -1
}

print(f"Новое количество признаков: {len(mega_features_v2)}")
print("Запуск долгого аккуратного обучения на GPU...")

model_v2 = lgb.train(
    params_v2,
    lgb_train_v2,
    num_boost_round=2500, # Увеличили раунды
    valid_sets=[lgb_train_v2, lgb_val_v2],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)

val_preds_v2 = model_v2.predict(X_val_v2)
rmse_v2 = np.sqrt(mean_squared_error(y_val_v2, val_preds_v2))
print(f"Новейший локальный RMSE: {rmse_v2:.5f}")

In [ ]:
print("Запуск экстраординарного инжиниринга признаков...")

# --- 1. ТЕКСТОВО-ВИЗУАЛЬНОЕ СВЯЗЫВАНИЕ (Cross-Features) ---
# Перемножаем главные компоненты текста и картинок, чтобы поймать несоответствие
# Например, если текст позитивный (высокий SVD), а картинка темная/плохая (низкий PCA)
for t_idx in range(3):
    for img_idx in range(3):
        df[f'cross_title_img_{t_idx}_{img_idx}'] = df[f'title_svd_adv_{t_idx}'] * df[f'img_pca_{img_idx}']
        df[f'cross_desc_img_{t_idx}_{img_idx}'] = df[f'desc_svd_adv_{t_idx}'] * df[f'img_pca_{img_idx}']

# --- 2. ЭФФЕКТ ЦЕНОВОГО ДАВЛЕНИЯ (Pricing Pressure) ---
# Насколько цена отклоняется от медианы в процентах — это мы делали.
# А теперь найдем Z-score (сигма-отклонение) цены внутри конкретного города и категории.
# Это подсветит аномально дешевые (мошенники/срочно) и дорогие (не купят) товары.

# Считаем среднее и стандартное отклонение для каждой группы city + category_name
group_stats = df.groupby(['city', 'category_name'])['price'].agg(['mean', 'std']).reset_index()
group_stats.columns = ['city', 'category_name', 'group_price_mean', 'group_price_std']

df = pd.merge(df, group_stats, on=['city', 'category_name'], how='left')
df['group_price_std'] = df['group_price_std'].fillna(0)

# Считаем Z-score: (цена - среднее) / std
df['price_z_score'] = (df['price'] - df['group_price_mean']) / (df['group_price_std'] + 1e-5)

# --- 3. СЧЁТЧИКИ УНИКАЛЬНОСТИ ОБЪЯВЛЕНИЯ (Uniqueness) ---
# Перекупщики часто копируют описания и цены. Посчитаем, сколько раз точная копия 
# этой цены встречается внутри этой же категории в тот же день.
df['date_str'] = df['activation_date'].dt.date.astype(str)
uniqueness_stats = df.groupby(['date_str', 'category_name', 'price']).size().reset_index(name='price_duplication_vibe')

df = pd.merge(df, uniqueness_stats, on=['date_str', 'category_name', 'price'], how='left')
df.drop(['date_str'], axis=1, inplace=True)

print("Экстраординарные признаки успешно сгенерированы!")

In [ ]:
# Собираем новые кросс-фичи и ценовые аномалии
extra_features = [f'cross_title_img_{t}_{i}' for t in range(3) for i in range(3)] + \
                 [f'cross_desc_img_{d}_{i}' for d in range(3) for i in range(3)] + \
                 ['price_z_score', 'price_duplication_vibe']

ultimate_features = mega_features_v2 + extra_features

# Пересобираем матрицы
X_train_ult = df.iloc[:train_len][ultimate_features]
y_train_ult = target

X_tr_u, X_val_u, y_tr_u, y_val_u = train_test_split(X_train_ult, y_train_ult, test_size=0.2, random_state=42)

lgb_train_u = lgb.Dataset(X_tr_u, y_tr_u)
lgb_val_u = lgb.Dataset(X_val_u, y_val_u, reference=lgb_train_u)

# Экстремальные параметры для финального штурма
params_ultimate = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.03,  # Еще плавнее шаг для точечной настройки
    'num_leaves': 127,      # Мощные, разветвленные деревья для 80+ признаков
    'feature_fraction': 0.7, # Берем 70% случайных фич (чтобы кросс-фичи не доминировали)
    'bagging_fraction': 0.8, # Беггинг датасета для стабильности
    'bagging_freq': 1,
    'seed': 42,
    'verbose': -1
}

print(f"Итого признаков на борту: {len(ultimate_features)}")
print("Запуск финального штурма на GPU...")

model_ultimate = lgb.train(
    params_ultimate,
    lgb_train_u,
    num_boost_round=3000,
    valid_sets=[lgb_train_u, lgb_val_u],
    callbacks=[lgb.early_stopping(stopping_rounds=70), lgb.log_evaluation(100)]
)

val_preds_u = model_ultimate.predict(X_val_u)
rmse_u = np.sqrt(mean_squared_error(y_val_u, val_preds_u))
print(f"Экстраординарный итоговый RMSE: {rmse_u:.5f}")

In [ ]:
print("Подготовка финального сабмита...")

# 1. Выделяем тестовую часть из нашего общего обработанного DataFrame
X_test_final = df.iloc[train_len:][ultimate_features]

# 2. Делаем предсказания нашей лучшей моделью
print("Генерация предсказаний для тестовой выборки...")
test_preds = model_ultimate.predict(X_test_final)

# 3. Жесткий постпроцессинг (Clip)
# Так как модель регрессионная, она чисто математически может выдать значения -0.02 или 1.05.
# Ограничиваем их строгими рамками [0, 1], чтобы не получить штраф на Kaggle
test_preds = np.clip(test_preds, 0.0, 1.0)

# 4. Загружаем шаблон сабмита и подменяем ответы
sub_path = os.path.join(PATH, 'sample_submission.csv')
submission = pd.read_csv(sub_path)
submission['deal_probability'] = test_preds

# 5. Сохраняем результат в текущую рабочую директорию Kaggle (/kaggle/working/)
submission.to_csv('submission.csv', index=False)

print("Файл submission.csv успешно создан и сохранен!")
print(submission.head())